# 04. Mercado de renda variável (Ibovespa)
Classe RendaVariavel: a média, a matriz de covariância e o sorteio dos cenários. Requisito F4.

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd

## Desenvolvimento

Mesma coisa do notebook anterior: a classe foi escrita aqui e depois copiada para app/mercado.py.

In [2]:
class RendaVariavel:
    """Mercado de renda variavel (Ibovespa): a distribuicao dos retornos. (F4)

    Trabalha com retornos liquidos, do mesmo jeito que estao na tabela
    retornos do banco. A diferenca R - R_f e montada depois, no agente.
    """

    def __init__(self, retornos: pd.DataFrame, coluna_data: str = "data") -> None:
        """
        Recebe o DataFrame de retornos, com uma coluna por ativo em decimal
        e sem valor faltando. A coluna de data e opcional; se existir, o nome
        dela vem em coluna_data e ela fica de fora das contas.
        """
        df = retornos.drop(columns=[coluna_data]) if coluna_data in retornos.columns else retornos.copy()
        self.ativos: list[str] = list(df.columns)
        self._R: np.ndarray = df.to_numpy(dtype=np.float64)  # (T, N)

        if self._R.ndim != 2 or self._R.shape[1] == 0:
            raise ValueError("retornos deve conter ao menos uma coluna de ativo.")
        if self._R.shape[0] < 2:
            raise ValueError("são necessárias ao menos 2 observações de retorno.")
        if np.isnan(self._R).any():
            raise ValueError("retornos não pode conter NaN.")

    @property
    def n_ativos(self) -> int:
        return self._R.shape[1]

    def media(self) -> np.ndarray:
        """Vetor de retornos esperados estimado mu_chapeu, shape (N,). (F2, F4)"""
        return self._R.mean(axis=0)

    def covariancia(self) -> np.ndarray:
        """A matriz de covariancia amostral, com ddof=1. (F2, F4)

        E o np.cov mesmo, com atleast_2d por cima pra garantir que o resultado
        seja bidimensional quando tem um ativo so.
        """
        return np.atleast_2d(np.cov(self._R.T, ddof=1))

    def amostrar(self, n: int, seed: int | None = None) -> np.ndarray:
        """
        Sorteia n cenarios de retorno de uma normal com a media e a
        covariancia estimadas.

        E o que alimenta o Monte Carlo da Etapa 1. Passando a mesma semente sai
        sempre o mesmo resultado, que e o que o NF4 pede.
        """
        rng = np.random.default_rng(seed)
        mu = self.media()
        # R = media + z * chol.T, com z normal padrao. Assim a covariancia sai certa.
        chol = np.linalg.cholesky(self.covariancia())
        z = rng.standard_normal((n, mu.shape[0]))
        return mu[None, :] + z @ chol.T

**Teste**: comparar a média e a covariância com o que o numpy dá, mais o n_ativos e o amostrar.

In [3]:
rng = np.random.default_rng(0)
dados = pd.DataFrame({'data': pd.date_range('2000-01', periods=120, freq='MS').strftime('%Y-%m'), 
                      'ibov': rng.normal(0.012, 0.05, 120), 'acao2': rng.normal(0.008, 0.04, 120)}) 
mv = RendaVariavel(dados)
R = dados[['ibov','acao2']].to_numpy() 

print('n_ativos:', mv.n_ativos, '| media:', mv.media()) 

n_ativos: 2 | media: [0.01606073 0.00295744]


In [4]:
assert np.allclose(mv.media(), R.mean(axis=0)) and np.allclose(mv.covariancia(), np.cov(R.T, ddof=1))

In [6]:
am = mv.amostrar(1_000_000, seed=1)
print('amostra media ~ mu_hat:', am.mean(axis=0))

amostra media ~ mu_hat: [0.01612577 0.00296028]


In [7]:
assert np.allclose(am.mean(axis=0), mv.media(), atol=1e-3)